In [26]:
import sys, os
from pathlib import Path
import pandas as pd
import numpy as np
import calendar
import time
from datetime import datetime
from dateutil.relativedelta import relativedelta
import warnings
warnings.filterwarnings('ignore')

def add_repo_path():
    here = Path.cwd()
    for p in [here, *here.parents]:
        if (p / "DATA").exists():
            if str(p) not in sys.path:
                sys.path.insert(0, str(p))
            return str(p)
    fallback = r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast"
    if os.path.isdir(fallback) and fallback not in sys.path:
        sys.path.insert(0, fallback)
    return fallback

project_path = add_repo_path()
print("Using project path:", project_path)

from DATA.stock_invest_function import *
import importlib
import DATA.us_sarima_forecast as sarima
importlib.reload(sarima)
import DATA.us_lstm_forecast_v2 as lstm_v2
importlib.reload(lstm_v2)
import DATA.us_prophet_forecast_v3 as prophet_v3
importlib.reload(prophet_v3)
import DATA.us_est_forecast_v2 as esmod
importlib.reload(esmod)

# 유틸리티 함수들
def convert_to_month_end(date_str):
    try:
        date_obj = pd.to_datetime(date_str)
        if pd.isna(date_obj):
            return None
        y, m, d = date_obj.year, date_obj.month, date_obj.day
        if 1 <= d <= 5:
            if m == 1:
                prev_y, prev_m = y - 1, 12
            else:
                prev_y, prev_m = y, m - 1
            last_day_prev = calendar.monthrange(prev_y, prev_m)[1]
            return datetime(prev_y, prev_m, last_day_prev)
        last_day_cur = calendar.monthrange(y, m)[1]
        return datetime(y, m, last_day_cur)
    except Exception:
        return None

def process_daily_to_monthly_market_data(daily_data, ticker):
    if not daily_data:
        return pd.DataFrame()
    df = pd.DataFrame(daily_data)
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date')
    df['year_month'] = df['date'].dt.to_period('M')
    monthly_data = []
    for year_month in df['year_month'].unique():
        month_data = df[df['year_month'] == year_month]
        last_day_data = month_data.loc[month_data['date'].idxmax()]
        monthly_data.append({
            'ticker': ticker,
            'date': last_day_data['date'],
            'market_cap': last_day_data['marketCap'],
            'market_cap_billions': round(last_day_data['marketCap'] / 1_000_000_000, 2),
        })
    return pd.DataFrame(monthly_data)

def fetch_revenue_data(ticker, api_key):
    url = f"https://financialmodelingprep.com/api/v3/income-statement/{ticker}"
    params = {'limit': 200, 'apikey': api_key, 'period': 'quarter'}
    try:
        response = requests.get(url, params=params, timeout=30)
        if response.status_code != 200:
            return None, f"HTTP {response.status_code}"
        data = response.json()
        if isinstance(data, dict) and 'Error Message' in data:
            return None, f"API 오류: {data['Error Message']}"
        if not data:
            return None, "데이터 없음"
        return data, None
    except Exception as e:
        return None, f"오류: {str(e)}"

def fetch_market_data_yearly(ticker, api_key, start_year=2010):
    all_data = []
    current_year = pd.Timestamp.now().year  # datetime 대신 pandas 사용
    for year in range(start_year, current_year + 1):
        start_date_str = f"{year}-01-01"
        end_date_str = f"{year}-12-31"
        url = f"https://financialmodelingprep.com/api/v3/historical-market-capitalization/{ticker}"
        params = {'from': start_date_str, 'to': end_date_str, 'apikey': api_key}
        try:
            response = requests.get(url, params=params, timeout=30)
            if response.status_code == 200:
                data = response.json()
                if data and isinstance(data, list):
                    all_data.extend(data)
            time.sleep(0.3)
        except Exception as e:
            continue
    return all_data if all_data else None, None

def fetch_db_revenue_data(ticker, db_info, end_date='2025-08-31'):
    try:
        engine = create_engine(
            f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
            f"{db_info['host']}:{db_info['port']}/{db_info['database']}"
        )
        query = f"""
        SELECT date, ticker, saleq
        FROM US_fundq
        WHERE ticker = '{ticker}'
        AND saleq IS NOT NULL
        AND date <= '{end_date}'
        ORDER BY date ASC
        """
        df = pd.read_sql(query, con=engine)
        engine.dispose()
        if not df.empty:
            df['date'] = pd.to_datetime(df['date'])
            df['revenue_billions'] = df['saleq'] / 1000
            # ★ date를 그대로 date_month_end로 사용 (convert 함수 사용 안함)
            df['date_month_end'] = pd.to_datetime(df['date'])
        return df[['ticker', 'date', 'date_month_end', 'revenue_billions']]
    except Exception as e:
        return pd.DataFrame()

def fetch_db_market_data(ticker, db_info, end_date='2024-12-31'):
    try:
        engine = create_engine(
            f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
            f"{db_info['host']}:{db_info['port']}/{db_info['database']}"
        )
        query = f"""
        SELECT date, ticker, me
        FROM US_fundm
        WHERE ticker = '{ticker}'
        AND me IS NOT NULL
        AND date <= '{end_date}'
        ORDER BY date ASC
        """
        df = pd.read_sql(query, con=engine)
        engine.dispose()
        if not df.empty:
            df['date'] = pd.to_datetime(df['date'])
            df['market_cap_billions'] = df['me'] / 1000
            df['date_month_end'] = df['date'].apply(convert_to_month_end)
        return df[['ticker', 'date', 'date_month_end', 'market_cap_billions']]
    except Exception as e:
        return pd.DataFrame()

def calculate_enhanced_ttm_and_psr(merged_data):
    """Calculate enhanced TTM and PSR"""
    df = merged_data.copy()
    df['date_month_end'] = pd.to_datetime(df['date_month_end'], errors='coerce')
    df = df.sort_values(['date_month_end']).reset_index(drop=True)
    df = df.sort_values(['ticker', 'date_month_end']).reset_index(drop=True)
    df['revenue_ttm'] = df.groupby('ticker')['revenue_billions'].rolling(window=4, min_periods=1).sum().reset_index(0, drop=True)
    df['revenue_ttm_billions'] = df['revenue_ttm']
    df['revenue_ttm_shift'] = df.groupby('ticker')['revenue_ttm_billions'].shift(2)
    df['PSR_ttm'] = df['market_cap_billions'] / df['revenue_ttm_shift']
    df['PSR_ttm'] = df['PSR_ttm'].replace([np.inf, -np.inf], np.nan)
    return df

def prepare_revenue_ttm(df, revenue_key="revenue_billions", min_periods=1):
    """Revenue TTM 계산"""
    d = df.copy()
    if 'date_month_end' not in d.columns:
        d = d.reset_index().rename(columns={'index': 'date_month_end'})
    d['date_month_end'] = pd.to_datetime(d['date_month_end'])

    if 'ticker' not in d.columns:
        raise ValueError("ticker 칼럼이 필요합니다.")

    rev_cols = [c for c in d.columns if revenue_key in c]
    if not rev_cols:
        raise ValueError(f"'{revenue_key}' 가 포함된 칼럼을 찾지 못했습니다.")

    uniq_tickers = d['ticker'].dropna().unique()
    if len(uniq_tickers) == 1:
        d['ticker'] = d['ticker'].ffill().bfill()
    else:
        d = d[~d['ticker'].isna()].copy()

    d = d.sort_values(['ticker', 'date_month_end']).reset_index(drop=True)

    row_mean = d[rev_cols].mean(axis=1, skipna=True)
    for c in rev_cols:
        d[c] = d[c].fillna(row_mean)

    for c in rev_cols:
        ttm_col = f"{c}_ttm"
        d[ttm_col] = (
            d.groupby('ticker', group_keys=False)[c]
             .rolling(window=4, min_periods=min_periods)
             .sum()
             .reset_index(level=0, drop=True)
        )

    d = d.set_index('date_month_end')
    return d

def clean_rev_data(rev_data):
    """
    1) 'revenue' 컬럼 값이 NaN인 행 제거
    2) (calendar_year, period) 중복 행 제거 (첫 번째 행만 유지)
    """
    required = ['revenue', 'calendar_year', 'period']
    missing = [c for c in required if c not in rev_data.columns]
    if missing:
        raise ValueError(f"필수 컬럼이 없습니다: {missing}")

    d = rev_data.copy()

    # 1) revenue NaN인 행 제거
    before = len(d)
    d = d[~d['revenue'].isna()].copy()
    removed_nan = before - len(d)

    # 2) (calendar_year, period) 중복 제거 — 첫 행 유지(현재 순서 기준)
    before2 = len(d)
    d = d.drop_duplicates(subset=['calendar_year', 'period'], keep='first').reset_index(drop=True)
    removed_dup = before2 - len(d)

    print(f"[clean_rev_data] removed rows → revenue NaN: {removed_nan}, duplicates: {removed_dup}")
    return d


def process_single_ticker(ticker, api_key, db_info, start_date_month, end_date_month, measurement_date):
    """단일 ticker 처리 (원본 코드 구조 유지)"""

    print(f"\n{'='*80}")
    print(f"처리 시작: {ticker}")
    print(f"{'='*80}")

    try:
        # 1. FMP 매출 데이터 수집
        print(f"\n[{ticker}] 1. FMP 매출 데이터 수집 중...")
        revenue_data, error = fetch_revenue_data(ticker, api_key)

        if revenue_data is None:
            print(f"[{ticker}] ERROR: FMP 매출 데이터 수집 실패 - {error}")
            return None

        # ★ FMP 원본 데이터 출력
        print(f"\n[{ticker}] ===== FMP 원본 데이터 (처음 5개) =====")
        for i, item in enumerate(revenue_data[:5]):
            print(f"  [{i}] date: {item.get('date')}, revenue: {item.get('revenue')}, "
                  f"calendarYear: {item.get('calendarYear')}, period: {item.get('period')}")
        print(f"[{ticker}] FMP 원본 총 {len(revenue_data)}건")

        all_revenue_data = []
        for item in revenue_data:
            all_revenue_data.append({
                'ticker': ticker,
                'date': item.get('date', ''),
                'calendar_year': item.get('calendarYear', ''),
                'period': item.get('period', ''),
                'revenue': item.get('revenue', 0) if item.get('revenue') is not None else 0,
                'revenue_billions': round((item.get('revenue', 0) or 0) / 1_000_000_000, 2),
            })

        fmp_revenue_df = pd.DataFrame(all_revenue_data)
        fmp_revenue_df['date'] = pd.to_datetime(fmp_revenue_df['date'])
        fmp_revenue_df = fmp_revenue_df.sort_values(['ticker', 'date'])

        # ★ 분기 데이터는 원본 date를 date_month_end로 사용 (convert 함수 사용 안함)
        fmp_revenue_df['date_month_end'] = fmp_revenue_df['date']

        print(f"\n[{ticker}] ===== 날짜 확인 (처음 10개) =====")
        for idx in range(min(10, len(fmp_revenue_df))):
            row = fmp_revenue_df.iloc[idx]
            print(f"  date: {row['date'].strftime('%Y-%m-%d')}, period: {row['period']}, revenue: {row['revenue_billions']}B")

        # ★ 중복 제거 전 상태 출력
        print(f"\n[{ticker}] 중복 제거 전: {len(fmp_revenue_df)}건")
        print(f"[{ticker}] date_month_end 유니크 값: {fmp_revenue_df['date_month_end'].nunique()}개")

        # ★ date_month_end 중복 확인
        dup_check = fmp_revenue_df.groupby('date_month_end').size()
        if (dup_check > 1).any():
            print(f"[{ticker}] WARNING: date_month_end 중복 발견!")
            print(dup_check[dup_check > 1].head(10))

        # date_month_end 기준 중복 제거 (첫 번째 행 유지) - 원본 코드 그대로
        fmp_revenue_df = fmp_revenue_df.drop_duplicates(subset=['date_month_end'], keep='first').reset_index(drop=True)

        print(f"[{ticker}] 중복 제거 후 FMP 매출 데이터: {len(fmp_revenue_df)}건")
        print(f"[{ticker}] FMP 데이터 컬럼: {fmp_revenue_df.columns.tolist()}")

        # 2. DB 매출 데이터 가져오기
        print(f"\n[{ticker}] 2. DB 매출 데이터 병합 중...")
        db_revenue_raw = fetch_db_revenue_data(ticker, db_info)
        print(f"[{ticker}] DB 원본 데이터: {len(db_revenue_raw)}건")

        if not db_revenue_raw.empty:
            # 연속된 중복값만 제거
            db_revenue_df = db_revenue_raw.loc[
                db_revenue_raw['revenue_billions'] != db_revenue_raw['revenue_billions'].shift()
            ]
            print(f"[{ticker}] DB 연속 중복 제거 후: {len(db_revenue_df)}건")
        else:
            db_revenue_df = pd.DataFrame()

        # ★ 병합 (outer join으로 모든 데이터 유지)
        mereged_rev_data = pd.merge(fmp_revenue_df, db_revenue_df, on=['ticker', 'date_month_end'], how='outer')
        print(f"[{ticker}] 병합 후 데이터: {len(mereged_rev_data)}건")
        print(f"[{ticker}] 병합 후 컬럼: {mereged_rev_data.columns.tolist()}")

        # 기간 필터링
        rev_data = mereged_rev_data[mereged_rev_data['date_month_end'] >= start_date_month].copy()
        print(f"[{ticker}] 기간 필터링 후 ({start_date_month} 이후): {len(rev_data)}건")

        # ★ _x, _y 컬럼 병합 (원본 코드와 동일)
        if 'revenue_billions_x' in rev_data.columns and 'revenue_billions_y' in rev_data.columns:
            print(f"[{ticker}] revenue_billions_x NaN 개수: {rev_data['revenue_billions_x'].isna().sum()}")
            print(f"[{ticker}] revenue_billions_y NaN 개수: {rev_data['revenue_billions_y'].isna().sum()}")

            # FMP 우선, DB로 보충 (원본 코드 그대로)
            rev_data['revenue_billions_x'] = rev_data['revenue_billions_x'].fillna(rev_data['revenue_billions_y'])

            print(f"[{ticker}] 병합 후 revenue_billions_x NaN 개수: {rev_data['revenue_billions_x'].isna().sum()}")

            # 컬럼 이름 변경 (원본 코드 그대로)
            rev_data.rename(columns={'revenue_billions_x': 'revenue_billions'}, inplace=True)
        elif 'revenue_billions_x' in rev_data.columns:
            # DB 데이터가 없는 경우
            rev_data.rename(columns={'revenue_billions_x': 'revenue_billions'}, inplace=True)

        print(f"[{ticker}] 최종 데이터 컬럼: {rev_data.columns.tolist()}")
        print(f"[{ticker}] 최종 데이터 건수 (정제 전): {len(rev_data)}건")

        # ★ clean_rev_data 적용 (원본 코드 그대로)
        rev_data = clean_rev_data(rev_data)

        # 기간 미달 체크 (44개 미만)
        if len(rev_data) < 44:
            print(f"[{ticker}] SKIP: 데이터 기간 미달 ({len(rev_data)}건 < 44건)")
            return None

        print(f"[{ticker}] 매출 데이터 충족: {len(rev_data)}건")

        # 3. 매출 예측 수행
        periods = 4

        sarima_df, results = sarima.run_sarima_prediction(
            rev_data,
            forecast_quarters=periods,
            exog_col=None
        )
        sarima_df = sarima_df.sort_values("date_month_end").set_index("date_month_end")

        lstm_raw_df, lstm_results_4q = lstm_v2.run_lstm_revenue_prediction(rev_data, ticker=ticker, prediction_quarters=4)
        lstm_df = lstm_raw_df.drop_duplicates(subset=['revenue_billions_lstm_forecast'], keep='last')

        prophet_raw_df, res_4q = prophet_v3.run_prophet_revenue_only(rev_data, ticker=ticker, prediction_quarters=4)

        es_raw_df, res_q4 = esmod.run_es_revenue_quarterly(rev_data, ticker=ticker, prediction_quarters=4)

        # 4. FMP 시가총액 데이터 수집
        print(f"[{ticker}] 2. FMP 시가총액 데이터 수집 중...")
        market_data, error = fetch_market_data_yearly(ticker, api_key, start_year=2010)

        if not market_data:
            print(f"[{ticker}] ERROR: FMP 시가총액 데이터 수집 실패")
            return None

        fmp_market_df = process_daily_to_monthly_market_data(market_data, ticker).copy()
        fmp_market_df['date_month_end'] = fmp_market_df['date'].apply(convert_to_month_end)
        fmp_market_df = (fmp_market_df
                         .drop_duplicates(subset=['date_month_end'])
                         .sort_values('date_month_end')
                         .reset_index(drop=True))

        print(f"[{ticker}] FMP 시가총액 데이터: {len(fmp_market_df)}건")

        # 5. DB 시가총액 병합
        db_market_df = fetch_db_market_data(ticker, db_info)

        if db_market_df.empty:
            print(f"[{ticker}] INFO: DB 시가총액 데이터 없음 → FMP 데이터만 사용")
            merged_market_df = fmp_market_df.copy()
            merged_market_df['market_cap_billions_from_db'] = np.nan
        else:
            if 'date_month_end' not in db_market_df.columns:
                if 'date' in db_market_df.columns:
                    db_market_df['date_month_end'] = db_market_df['date'].apply(convert_to_month_end)
                else:
                    db_market_df = pd.DataFrame()

            if not db_market_df.empty:
                if 'market_cap_billions' in db_market_df.columns:
                    db_market_df_renamed = db_market_df.rename(
                        columns={'market_cap_billions': 'market_cap_billions_from_db'}
                    )
                else:
                    db_market_df_renamed = db_market_df[['date_month_end']].copy()
                    db_market_df_renamed['market_cap_billions_from_db'] = np.nan

                merged_market_df = fmp_market_df.merge(
                    db_market_df_renamed[['date_month_end', 'market_cap_billions_from_db']],
                    on='date_month_end',
                    how='left'
                )
            else:
                merged_market_df = fmp_market_df.copy()
                merged_market_df['market_cap_billions_from_db'] = np.nan

        if 'market_cap_billions' not in merged_market_df.columns:
            merged_market_df['market_cap_billions'] = np.nan

        if 'market_cap_billions_from_db' not in merged_market_df.columns:
            merged_market_df['market_cap_billions_from_db'] = np.nan

        merged_market_df['market_cap_billions'] = merged_market_df['market_cap_billions'].fillna(
            merged_market_df['market_cap_billions_from_db']
        )

        merged_market_df = (merged_market_df
                            .drop_duplicates(subset=['date_month_end'])
                            .sort_values('date_month_end')
                            .reset_index(drop=True))

        print(f"[{ticker}] 병합 완료: {len(merged_market_df)}건 (FMP+DB)")

        # 6. TTM 및 PSR 계산
        enhanced_merged_df = pd.merge(merged_market_df[['date_month_end', 'market_cap_billions']], rev_data, on='date_month_end', how='outer')
        market_cap_resize = enhanced_merged_df[['date_month_end', 'market_cap_billions', 'ticker', 'revenue_billions']].copy()
        market_cap_resize.dropna(subset=['market_cap_billions'], inplace=True)
        market_cap_resize.ffill(limit=2, inplace=True)
        market_cap_resize = market_cap_resize[(market_cap_resize['date_month_end'] >= start_date_month) & (market_cap_resize['date_month_end'] <= end_date_month)]
        market_cap_resize = market_cap_resize.dropna(axis=0)

        enhanced_merged_df_with_ttm = calculate_enhanced_ttm_and_psr(market_cap_resize)

        # 7. PSR 예측
        psr_sarima_df, psr_12_res = sarima.run_sarima_psr_only(
            df=enhanced_merged_df_with_ttm,
            periods=12,
            target_col="PSR_ttm",
            analysis_start="2012-06-01",
            warmup_months=6,
            fill_method="interpolate",
            ic="aic"
        )

        psr_lstm_df, psr_results = lstm_v2.run_lstm_psr_prediction(enhanced_merged_df_with_ttm, ticker=ticker, prediction_months=12)

        psr_prophet_df, psr_res = prophet_v3.run_prophet_psr_only(enhanced_merged_df_with_ttm, ticker=ticker, prediction_months=12)

        psr_es_df, psr_res_es = esmod.run_es_psr_only(
            df=enhanced_merged_df_with_ttm,
            ticker=ticker,
            prediction_months=12,
            start_date=None
        )

        # 8. Valuation 종합
        sarima_resize_df = sarima_df[['ticker', 'revenue_billions_sarima_noexog']].copy()
        lstm_resize_df = lstm_df[['revenue_billions_lstm_forecast']].copy()
        prophet_resize_df = prophet_raw_df[['revenue_billions_prophet_forecast']].copy()
        es_resize_df = es_raw_df[['revenue_billions_esq_forecast']].copy()

        revenue_forecast_df = pd.concat([sarima_resize_df, lstm_resize_df, prophet_resize_df, es_resize_df], axis=1)

        psr_sarima_resiae = psr_sarima_df[['PSR_ttm_sarima_forecast']]
        psr_lstm_resiae = psr_lstm_df[['PSR_ttm_lstm_forecast']]
        psr_prophet_resiae = psr_prophet_df[['PSR_prophet_forecast_noexog']]
        psr_es_resiae = psr_es_df[['PSR_es_forecast']]

        psr_forecast_df = pd.concat([psr_sarima_resiae, psr_lstm_resiae, psr_prophet_resiae, psr_es_resiae], axis=1)

        revenue_forecast_ = prepare_revenue_ttm(revenue_forecast_df)
        revenue_forecast_ttm = revenue_forecast_.filter(like='_ttm')
        revenue_forecast_ttm['ticker'] = ticker

        valuation_df = pd.concat([revenue_forecast_ttm, psr_forecast_df], axis=1)

        # 9. Valuation 계산
        valuation_filled = valuation_df.copy()

        cols_to_fill = ['ticker'] + [c for c in valuation_filled.columns if 'revenue' in c]
        valuation_filled[cols_to_fill] = valuation_filled[cols_to_fill].ffill(limit=2)

        valuation_filled['sarima_valuation'] = (
            valuation_filled['revenue_billions_sarima_noexog_ttm'] *
            valuation_filled['PSR_ttm_sarima_forecast']
        )

        valuation_filled['lstm_valuation'] = (
            valuation_filled['revenue_billions_lstm_forecast_ttm'] *
            valuation_filled['PSR_ttm_lstm_forecast']
        )

        valuation_filled['prophet_valuation'] = (
            valuation_filled['revenue_billions_prophet_forecast_ttm'] *
            valuation_filled['PSR_prophet_forecast_noexog']
        )

        valuation_filled['es_valuation'] = (
            valuation_filled['revenue_billions_esq_forecast_ttm'] *
            valuation_filled['PSR_es_forecast']
        )

        # 10. 마지막 15개월 추출
        if 'date_month_end' in valuation_filled.columns:
            valuation_filled = valuation_filled.sort_values('date_month_end')
            valuation_result = valuation_filled.groupby('ticker').tail(15).reset_index(drop=True)
        else:
            valuation_filled = valuation_filled.sort_index()
            valuation_result = valuation_filled.groupby('ticker').tail(15).reset_index()

        # 11. 측정 날짜 추가
        valuation_result['measurement_date'] = measurement_date

        print(f"[{ticker}] SUCCESS: Valuation 계산 완료 ({len(valuation_result)}건)")
        return valuation_result

    except Exception as e:
        print(f"[{ticker}] ERROR: {str(e)}")
        import traceback
        traceback.print_exc()
        return None


def process_multiple_tickers(ticker_list, api_key, db_info, start_date_month, end_date_month, measurement_date=None):
    """여러 ticker 순차 처리"""

    if measurement_date is None:
        measurement_date = pd.Timestamp.today().strftime('%Y-%m-%d')

    print(f"\n{'='*80}")
    print(f"Multi-Ticker Valuation 시작")
    print(f"총 {len(ticker_list)}개 종목")
    print(f"기간: {start_date_month} ~ {end_date_month}")
    print(f"측정일: {measurement_date}")
    print(f"{'='*80}\n")

    all_results = []
    success_count = 0
    fail_count = 0

    for idx, ticker in enumerate(ticker_list, 1):
        print(f"\n[{idx}/{len(ticker_list)}] 처리 중: {ticker}")

        result = process_single_ticker(
            ticker=ticker,
            api_key=api_key,
            db_info=db_info,
            start_date_month=start_date_month,
            end_date_month=end_date_month,
            measurement_date=measurement_date
        )

        if result is not None:
            all_results.append(result)
            success_count += 1
            print(f"[{ticker}] ✓ 성공")
        else:
            fail_count += 1
            print(f"[{ticker}] ✗ 실패")

    print(f"\n{'='*80}")
    print(f"처리 완료:")
    print(f"  성공: {success_count}개")
    print(f"  실패: {fail_count}개")
    print(f"{'='*80}\n")

    if not all_results:
        print("WARNING: 성공한 결과가 없습니다.")
        return pd.DataFrame()

    # 모든 결과 결합
    combined_df = pd.concat(all_results, ignore_index=True)

    return combined_df

Using project path: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy


In [27]:
# ========================
# 실행 예시
# ========================

if __name__ == "__main__":
    # 설정
    ticker_list = ['AAPL', 'MSFT', 'VVV']  # 처리할 ticker 리스트

    api_key = 'hT0gAk87j9xZx4PlBApvBqfVL5IahvgV'

    db_info = {
        'host': get_db_host(),
        'port': 3307,
        'user': 'stox7412',
        'password': 'Apt106503!~',
        'database': 'investar'
    }

    start_date_month = '2011-03-01'
    end_date_month = (pd.Timestamp.today().normalize() - pd.offsets.MonthEnd(1)).strftime('%Y-%m-%d')
    measurement_date = pd.Timestamp.today().strftime('%Y-%m-%d')

    # 멀티 ticker 처리
    final_results = process_multiple_tickers(
        ticker_list=ticker_list,
        api_key=api_key,
        db_info=db_info,
        start_date_month=start_date_month,
        end_date_month=end_date_month,
        measurement_date=measurement_date
    )

    # 결과 확인 및 저장
    if not final_results.empty:
        print("\n최종 결과:")
        print(final_results.head(20))
        print(f"\n총 {len(final_results)}개 행")

        # CSV 저장
        output_file = f"valuation_results_{pd.Timestamp.today().strftime('%Y%m%d')}.csv"
        final_results.to_csv(output_file, index=False, encoding='utf-8-sig')
        print(f"\n결과 저장: {output_file}")
    else:
        print("\n결과가 없습니다.")


Multi-Ticker Valuation 시작
총 3개 종목
기간: 2011-03-01 ~ 2025-09-30
측정일: 2025-10-06


[1/3] 처리 중: AAPL

처리 시작: AAPL

[AAPL] 1. FMP 매출 데이터 수집 중...

[AAPL] ===== FMP 원본 데이터 (처음 5개) =====
  [0] date: 2025-06-28, revenue: 94036000000, calendarYear: 2025, period: Q3
  [1] date: 2025-03-29, revenue: 95359000000, calendarYear: 2025, period: Q2
  [2] date: 2024-12-28, revenue: 124300000000, calendarYear: 2025, period: Q1
  [3] date: 2024-09-28, revenue: 94930000000, calendarYear: 2024, period: Q4
  [4] date: 2024-06-29, revenue: 85777000000, calendarYear: 2024, period: Q3
[AAPL] FMP 원본 총 160건

[AAPL] ===== 날짜 확인 (처음 10개) =====
  date: 1985-09-30, period: Q4, revenue: 0.41B
  date: 1985-12-31, period: Q1, revenue: 0.53B
  date: 1986-03-31, period: Q2, revenue: 0.41B
  date: 1986-06-30, period: Q3, revenue: 0.45B
  date: 1986-09-30, period: Q4, revenue: 0.51B
  date: 1986-12-31, period: Q1, revenue: 0.66B
  date: 1987-03-31, period: Q2, revenue: 0.58B
  date: 1987-06-30, period: Q3, revenue: 0.64B
  

01:59:27 - cmdstanpy - INFO - Chain [1] start processing
01:59:27 - cmdstanpy - INFO - Chain [1] done processing


[AAPL] 2. FMP 시가총액 데이터 수집 중...
[AAPL] FMP 시가총액 데이터: 1건
[AAPL] 병합 완료: 1건 (FMP+DB)
[AAPL] ERROR: You are trying to merge on object and datetime64[ns] columns for key 'date_month_end'. If you wish to proceed you should use pd.concat
[AAPL] ✗ 실패

[2/3] 처리 중: MSFT

처리 시작: MSFT

[MSFT] 1. FMP 매출 데이터 수집 중...


Traceback (most recent call last):
  File "C:\Users\82108\AppData\Local\Temp\ipykernel_7392\603573673.py", line 435, in process_single_ticker
    enhanced_merged_df = pd.merge(merged_market_df[['date_month_end', 'market_cap_billions']], rev_data, on='date_month_end', how='outer')
  File "C:\Users\82108\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\reshape\merge.py", line 170, in merge
    op = _MergeOperation(
  File "C:\Users\82108\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\reshape\merge.py", line 807, in __init__
    self._maybe_coerce_merge_keys()
  File "C:\Users\82108\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\reshape\merge.py", line 1514, in _maybe_coerce_merge_keys
    raise ValueError(msg)
ValueError: You are trying to merge on object and datetime64[ns] columns for key 'date_month_end'. If you wish to proceed you should use pd.concat



[MSFT] ===== FMP 원본 데이터 (처음 5개) =====
  [0] date: 2025-06-30, revenue: 76441000000, calendarYear: 2025, period: Q4
  [1] date: 2025-03-31, revenue: 70066000000, calendarYear: 2025, period: Q3
  [2] date: 2024-12-31, revenue: 69632000000, calendarYear: 2025, period: Q2
  [3] date: 2024-09-30, revenue: 65585000000, calendarYear: 2025, period: Q1
  [4] date: 2024-06-30, revenue: 64727000000, calendarYear: 2024, period: Q4
[MSFT] FMP 원본 총 160건

[MSFT] ===== 날짜 확인 (처음 10개) =====
  date: 1985-09-30, period: Q1, revenue: 0.04B
  date: 1985-12-31, period: Q2, revenue: 0.04B
  date: 1986-03-31, period: Q3, revenue: 0.05B
  date: 1986-06-30, period: Q4, revenue: 0.06B
  date: 1986-09-30, period: Q1, revenue: 0.07B
  date: 1986-12-31, period: Q2, revenue: 0.08B
  date: 1987-03-31, period: Q3, revenue: 0.1B
  date: 1987-06-30, period: Q4, revenue: 0.1B
  date: 1987-09-30, period: Q1, revenue: 0.1B
  date: 1987-12-31, period: Q2, revenue: 0.16B

[MSFT] 중복 제거 전: 160건
[MSFT] date_month_end 유니크 값: 16

02:00:02 - cmdstanpy - INFO - Chain [1] start processing
02:00:03 - cmdstanpy - INFO - Chain [1] done processing


[MSFT] 2. FMP 시가총액 데이터 수집 중...
[MSFT] FMP 시가총액 데이터: 1건
[MSFT] INFO: DB 시가총액 데이터 없음 → FMP 데이터만 사용
[MSFT] 병합 완료: 1건 (FMP+DB)
[MSFT] ERROR: You are trying to merge on object and datetime64[ns] columns for key 'date_month_end'. If you wish to proceed you should use pd.concat


Traceback (most recent call last):
  File "C:\Users\82108\AppData\Local\Temp\ipykernel_7392\603573673.py", line 435, in process_single_ticker
    enhanced_merged_df = pd.merge(merged_market_df[['date_month_end', 'market_cap_billions']], rev_data, on='date_month_end', how='outer')
  File "C:\Users\82108\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\reshape\merge.py", line 170, in merge
    op = _MergeOperation(
  File "C:\Users\82108\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\reshape\merge.py", line 807, in __init__
    self._maybe_coerce_merge_keys()
  File "C:\Users\82108\AppData\Local\Programs\Python\Python39\lib\site-packages\pandas\core\reshape\merge.py", line 1514, in _maybe_coerce_merge_keys
    raise ValueError(msg)
ValueError: You are trying to merge on object and datetime64[ns] columns for key 'date_month_end'. If you wish to proceed you should use pd.concat


[MSFT] ✗ 실패

[3/3] 처리 중: VVV

처리 시작: VVV

[VVV] 1. FMP 매출 데이터 수집 중...

[VVV] ===== FMP 원본 데이터 (처음 5개) =====
  [0] date: 2025-06-30, revenue: 439000000, calendarYear: 2025, period: Q3
  [1] date: 2025-03-31, revenue: 403200000, calendarYear: 2025, period: Q2
  [2] date: 2024-12-31, revenue: 414300000, calendarYear: 2025, period: Q1
  [3] date: 2024-09-30, revenue: 435500000, calendarYear: 2024, period: Q4
  [4] date: 2024-06-30, revenue: 421400000, calendarYear: 2024, period: Q3
[VVV] FMP 원본 총 43건

[VVV] ===== 날짜 확인 (처음 10개) =====
  date: 2014-12-31, period: Q1, revenue: 0.49B
  date: 2015-03-31, period: Q2, revenue: 0.49B
  date: 2015-06-30, period: Q3, revenue: 0.51B
  date: 2015-09-30, period: Q4, revenue: 0.48B
  date: 2015-12-31, period: Q1, revenue: 0.46B
  date: 2016-03-31, period: Q2, revenue: 0.48B
  date: 2016-06-30, period: Q3, revenue: 0.5B
  date: 2016-09-30, period: Q4, revenue: 0.49B
  date: 2016-12-31, period: Q1, revenue: 0.49B
  date: 2017-03-31, period: Q2, revenue: 0

In [29]:
final_results

""


In [31]:
# ==========================================================
# 1) 단일 티커 처리 함수 (원본 실행부를 최대한 그대로 감싸기)
# ==========================================================
def run_valuation_for_ticker(
    one_ticker: str,
    api_key: str,
    db_info: dict,
    start_date_month: str,
    end_date_month: str,
    periods: int = 4,
    min_rows_required: int = 44,
) -> Optional[pd.DataFrame]:
    """
    - rev_data 길이가 44행 미만이면 pass
    - 성공 시 valuation_result(DataFrame) 반환
    - 실패 시 None 반환
    - measured_at(측정일) 칼럼 추가
    """
    try:
        ticker = one_ticker  # 🔒 원본 코드 내부 변수명 유지

        print("=" * 80)
        print("전처리 과정 테스트 시작")
        print(f"대상 종목: {ticker}")
        print("=" * 80)

        # 1. FMP 매출 데이터 수집
        print("\n1. FMP 매출 데이터 수집 중...")
        revenue_data, error = fetch_revenue_data(ticker, api_key)

        if revenue_data is None:
            print(f"ERROR: FMP 매출 데이터 수집 실패 - {error}")
            return None  # ⬅️ pass

        all_revenue_data = []
        for item in revenue_data:
            all_revenue_data.append({
                'ticker': ticker,
                'date': item.get('date', ''),
                'calendar_year': item.get('calendarYear', ''),
                'period': item.get('period', ''),
                'revenue': item.get('revenue', 0) if item.get('revenue') is not None else 0,
                'revenue_billions': round((item.get('revenue', 0) or 0) / 1_000_000_000, 2),
            })

        fmp_revenue_df = pd.DataFrame(all_revenue_data)
        fmp_revenue_df['date'] = pd.to_datetime(fmp_revenue_df['date'])
        fmp_revenue_df = fmp_revenue_df.sort_values(['ticker', 'date'])
        fmp_revenue_df['date_month_end'] = fmp_revenue_df['date'].apply(convert_to_month_end)
        fmp_revenue_df = fmp_revenue_df.drop_duplicates(subset=['date_month_end'], keep='first').reset_index(drop=True)
        print(f"FMP 매출 데이터: {len(fmp_revenue_df)}건")

        # 2. DB 매출 데이터 가져오기
        db_revenue_raw = fetch_db_revenue_data(ticker, db_info)
        db_revenue_df = db_revenue_raw.loc[db_revenue_raw['revenue_billions'] != db_revenue_raw['revenue_billions'].shift()]

        # 병합
        mereged_rev_data = pd.merge(fmp_revenue_df, db_revenue_df, on=['ticker', 'date_month_end'], how='outer')
        rev_data = mereged_rev_data[mereged_rev_data['date_month_end'] >= start_date_month]
        rev_data['revenue_billions_x'] = rev_data['revenue_billions_x'].fillna(rev_data['revenue_billions_y'])
        rev_data.rename(columns={'revenue_billions_x': 'revenue_billions'}, inplace=True)

        # 🔎 rev_data 길이 체크(원하신 44 미만 pass)
        if len(rev_data) < min_rows_required:
            print(f"[{ticker}] rev_data rows={len(rev_data)} < {min_rows_required} → pass")
            return None

        rev_data = clean_rev_data(rev_data)

        # periods=4 또는 8 등 원하는 분기 수
        # periods = 4  # ← 인자에서 받으므로 주석
        # 모듈 함수 호출 (내부에서 월말 정렬/중복제거 처리)
        sarima_df, results = sarima.run_sarima_prediction(
            rev_data,
            forecast_quarters=periods,   # ← 예측 분기 수
            exog_col=None                # 외생변수 없으면 None
        )

        # 인덱스를 date_month_end로 설정
        sarima_df = sarima_df.sort_values("date_month_end").set_index("date_month_end")

        # 1) 4분기 예측
        lstm_raw_df, lstm_results_4q = lstm_v2.run_lstm_revenue_prediction(rev_data, ticker=ticker, prediction_quarters=4)
        lstm_df = lstm_raw_df.drop_duplicates(subset=['revenue_billions_lstm_forecast'], keep='last')

        prophet_raw_df, res_4q = prophet_v3.run_prophet_revenue_only(rev_data, ticker=ticker, prediction_quarters=4)

        es_raw_df, res_q4 = esmod.run_es_revenue_quarterly(rev_data, ticker=ticker, prediction_quarters=4)

        # 3. FMP 시가총액 데이터 수집
        print("2. FMP 시가총액 데이터 수집 중...")
        market_data, error = fetch_market_data_yearly(ticker, api_key, start_year=2010)

        if not market_data:
            print("ERROR: FMP 시가총액 데이터 수집 실패")
            return None  # ⬅️ pass

        fmp_market_df = process_daily_to_monthly_market_data(market_data, ticker).copy()
        fmp_market_df['date_month_end'] = fmp_market_df['date'].apply(convert_to_month_end)
        fmp_market_df = (fmp_market_df
                         .drop_duplicates(subset=['date_month_end'])
                         .sort_values('date_month_end')
                         .reset_index(drop=True))

        print(f"FMP 시가총액 데이터: {len(fmp_market_df)}건")

        # -----------------------------
        # 안전 병합: DB가 없으면 FMP만 사용
        # -----------------------------
        def _safe_get_db_market_df():
            try:
                df = fetch_db_market_data(ticker, db_info)
                if df is None or len(df) == 0:
                    return pd.DataFrame()
                return df.copy()
            except Exception as e:
                print(f"[WARN] DB 조회 중 예외 발생: {e}")
                return pd.DataFrame()

        db_market_df = _safe_get_db_market_df()

        if not db_market_df.empty:
            if 'date_month_end' not in db_market_df.columns:
                if 'date' in db_market_df.columns:
                    db_market_df['date_month_end'] = db_market_df['date'].apply(convert_to_month_end)
                else:
                    print("[WARN] DB 데이터에 날짜 컬럼이 없어 병합을 건너뜁니다.")
                    db_market_df = pd.DataFrame()

        if db_market_df.empty:
            print("[INFO] DB 시가총액 데이터 없음 → FMP 데이터만 사용합니다.")
            merged_market_df = fmp_market_df.copy()
            merged_market_df['market_cap_billions_from_db'] = np.nan
        else:
            if 'market_cap_billions' in db_market_df.columns:
                db_market_df_renamed = db_market_df.rename(
                    columns={'market_cap_billions': 'market_cap_billions_from_db'}
                )
            else:
                db_market_df_renamed = db_market_df[['date_month_end']].copy()
                db_market_df_renamed['market_cap_billions_from_db'] = np.nan
                print("[WARN] DB에 'market_cap_billions' 컬럼이 없어 NaN으로 채웁니다.")

            merged_market_df = fmp_market_df.merge(
                db_market_df_renamed[['date_month_end', 'market_cap_billions_from_db']],
                on='date_month_end',
                how='left'
            )

        if 'market_cap_billions' not in merged_market_df.columns:
            merged_market_df['market_cap_billions'] = np.nan
        if 'market_cap_billions_from_db' not in merged_market_df.columns:
            merged_market_df['market_cap_billions_from_db'] = np.nan

        merged_market_df['market_cap_billions'] = merged_market_df['market_cap_billions'].fillna(
            merged_market_df['market_cap_billions_from_db']
        )

        merged_market_df = (merged_market_df
                            .drop_duplicates(subset=['date_month_end'])
                            .sort_values('date_month_end')
                            .reset_index(drop=True))

        print(f"병합 완료: {len(merged_market_df)}건 (FMP+DB)")

        enhanced_merged_df = pd.merge(
            merged_market_df[['date_month_end', 'market_cap_billions']],
            rev_data, on='date_month_end', how='outer'
        )
        market_cap_resize = enhanced_merged_df[['date_month_end', 'market_cap_billions', 'ticker', 'revenue_billions']].copy()
        market_cap_resize.dropna(subset=['market_cap_billions'], inplace=True)
        market_cap_resize.ffill(limit=2, inplace=True)
        market_cap_resize = market_cap_resize[
            (market_cap_resize['date_month_end'] >= start_date_month) &
            (market_cap_resize['date_month_end'] <= end_date_month)
        ]
        market_cap_resize = market_cap_resize.dropna(axis=0)

        enhanced_merged_df_with_ttm = calculate_enhanced_ttm_and_psr(market_cap_resize)

        from DATA.us_sarima_forecast import run_sarima_psr_only

        # 12개월 예측 (PSR)
        psr_sarima_df, psr_12_res = run_sarima_psr_only(
            df=enhanced_merged_df_with_ttm,
            periods=12,
            target_col="PSR_ttm",
            analysis_start="2012-06-01",
            warmup_months=6,
            fill_method="interpolate",
            ic="aic"
        )

        psr_lstm_df, psr_results = lstm_v2.run_lstm_psr_prediction(
            enhanced_merged_df_with_ttm, ticker=ticker, prediction_months=12
        )

        psr_prophet_df, psr_res = prophet_v3.run_prophet_psr_only(
            enhanced_merged_df_with_ttm, ticker=ticker, prediction_months=12
        )

        psr_es_df, psr_res_es = esmod.run_es_psr_only(
            df=enhanced_merged_df_with_ttm,
            ticker=ticker,
            prediction_months=12,
            start_date=None
        )

        # ===== Valuation 종합 (원본 유지) =====
        sarima_resize_df = sarima_df[['ticker', 'revenue_billions_sarima_noexog']].copy()
        lstm_resize_df = lstm_df[['revenue_billions_lstm_forecast']].copy()
        prophet_resize_df = prophet_raw_df[['revenue_billions_prophet_forecast']].copy()
        es_resize_df = es_raw_df[['revenue_billions_esq_forecast']].copy()

        revenue_forecast_df = pd.concat([sarima_resize_df, lstm_resize_df, prophet_resize_df, es_resize_df], axis=1)

        psr_sarima_resiae = psr_sarima_df[['PSR_ttm_sarima_forecast']]
        psr_lstm_resiae = psr_lstm_df[['PSR_ttm_lstm_forecast']]
        psr_prophet_resiae = psr_prophet_df[['PSR_prophet_forecast_noexog']]
        psr_es_resiae = psr_es_df[['PSR_es_forecast']]

        psr_forecast_df = pd.concat([psr_sarima_resiae, psr_lstm_resiae, psr_prophet_resiae, psr_es_resiae], axis=1)

        revenue_forecast_ = prepare_revenue_ttm(revenue_forecast_df)
        revenue_forecast_ttm = revenue_forecast_.filter(like='_ttm')
        revenue_forecast_ttm['ticker'] = ticker

        valuation_df = pd.concat([revenue_forecast_ttm, psr_forecast_df], axis=1)

        # 1) 복사본 생성 (원본 보호)
        valuation_filled = valuation_df.copy()

        # 2) ffill 대상 칼럼 목록 생성
        cols_to_fill = ['ticker'] + [c for c in valuation_filled.columns if 'revenue' in c]

        # 3) 선택된 칼럼만 ffill(limit=2)
        valuation_filled[cols_to_fill] = valuation_filled[cols_to_fill].ffill(limit=2)

        required_cols = valuation_filled.columns.tolist()
        missing = [c for c in required_cols if c not in valuation_filled.columns]
        if missing:
            raise ValueError(f"다음 칼럼이 없습니다: {missing}")

        # 2. Valuation 계산 (revenue × PSR)
        valuation_filled['sarima_valuation'] = (
            valuation_filled['revenue_billions_sarima_noexog_ttm'] *
            valuation_filled['PSR_ttm_sarima_forecast']
        )
        valuation_filled['lstm_valuation'] = (
            valuation_filled['revenue_billions_lstm_forecast_ttm'] *
            valuation_filled['PSR_ttm_lstm_forecast']
        )
        valuation_filled['prophet_valuation'] = (
            valuation_filled['revenue_billions_prophet_forecast_ttm'] *
            valuation_filled['PSR_prophet_forecast_noexog']
        )
        valuation_filled['es_valuation'] = (
            valuation_filled['revenue_billions_esq_forecast_ttm'] *
            valuation_filled['PSR_es_forecast']
        )

        # 3. 마지막 15개월 추출
        if 'date_month_end' in valuation_filled.columns:
            valuation_filled = valuation_filled.sort_values('date_month_end')
            valuation_result = valuation_filled.groupby('ticker').tail(15).reset_index(drop=True)
        else:
            valuation_filled = valuation_filled.sort_index()
            valuation_result = valuation_filled.groupby('ticker').tail(15).reset_index()

        # 4) 측정일 추가
        valuation_result['measured_at'] = pd.Timestamp.today().normalize()

        return valuation_result

    except Exception as e:
        print(f"[{one_ticker}] ERROR: {e}")
        return None

  # ==========================================================
# 2) 여러 티커 일괄 실행 & 결과 결합
# ==========================================================
def run_batch_valuation(
    tickers: List[str],
    api_key: str,
    db_info: dict,
    start_date_month: str,
    end_date_month: str,
    periods: int = 4,
    min_rows_required: int = 44,
) -> pd.DataFrame:
    results = []
    for t in tickers:
        out = run_valuation_for_ticker(
            one_ticker=t,
            api_key=api_key,
            db_info=db_info,
            start_date_month=start_date_month,
            end_date_month=end_date_month,
            periods=periods,
            min_rows_required=min_rows_required
        )
        if out is None or out.empty:
            print(f"[{t}] skipped.")
            continue
        results.append(out)

    if not results:
        return pd.DataFrame()

    final_df = pd.concat(results, ignore_index=True)
    return final_df

In [32]:
# ==========================================================
# 3) 실행 예시
# ==========================================================
# 원하는 티커 목록
tickers = ['VVV', 'MU', 'ANET', 'AAPL', 'MMM', 'CAT']
final_valuation_df = run_batch_valuation(
    tickers=tickers,
    api_key=api_key,
    db_info=db_info,
    start_date_month=start_date_month,
    end_date_month=end_date_month,
    periods=4,
    min_rows_required=44,
)
print(final_valuation_df.tail())
final_valuation_df.to_parquet("valuation_results.parquet", index=False)

전처리 과정 테스트 시작
대상 종목: VVV

1. FMP 매출 데이터 수집 중...
FMP 매출 데이터: 1건
[VVV] ERROR: You are trying to merge on object and datetime64[ns] columns for key 'date_month_end'. If you wish to proceed you should use pd.concat
[VVV] skipped.
전처리 과정 테스트 시작
대상 종목: MU

1. FMP 매출 데이터 수집 중...
FMP 매출 데이터: 1건
[MU] ERROR: You are trying to merge on object and datetime64[ns] columns for key 'date_month_end'. If you wish to proceed you should use pd.concat
[MU] skipped.
전처리 과정 테스트 시작
대상 종목: ANET

1. FMP 매출 데이터 수집 중...
FMP 매출 데이터: 1건
[ANET] ERROR: You are trying to merge on object and datetime64[ns] columns for key 'date_month_end'. If you wish to proceed you should use pd.concat
[ANET] skipped.
전처리 과정 테스트 시작
대상 종목: AAPL

1. FMP 매출 데이터 수집 중...
FMP 매출 데이터: 1건
[AAPL] ERROR: You are trying to merge on object and datetime64[ns] columns for key 'date_month_end'. If you wish to proceed you should use pd.concat
[AAPL] skipped.
전처리 과정 테스트 시작
대상 종목: MMM

1. FMP 매출 데이터 수집 중...
FMP 매출 데이터: 1건
[MMM] ERROR: You are trying to

ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.

In [4]:
# repo 경로 추가 (이미 있으시면 그대로 사용)
project_path = add_repo_path()
print("Using project path:", project_path)

# ★ 핵심: stock_invest_function을 명시적으로 import + reload
import importlib
import DATA.stock_invest_function as sif
importlib.reload(sif)

# ★ 필요한 함수들만 정확한 이름으로 임포트
from DATA.stock_invest_function import (
    get_db_host,
    fetch_revenue_data,
    fetch_db_revenue_data,
    fetch_market_data_yearly,
    fetch_db_market_data,
    process_daily_to_monthly_market_data,
)

# (선택) 정상 로드 확인용 — 문제시 바로 알림
assert callable(fetch_revenue_data), "fetch_revenue_data import 실패"
assert callable(fetch_db_revenue_data), "fetch_db_revenue_data import 실패"
assert callable(fetch_market_data_yearly), "fetch_market_data_yearly import 실패"
assert callable(fetch_db_market_data), "fetch_db_market_data import 실패"
assert callable(process_daily_to_monthly_market_data), "process_daily_to_monthly_market_data import 실패"


db_info = {
    'host': get_db_host(),
    # 'host': '192.168.0.230',
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}


# ====== 사용 예시 ======
tickers = ['VVV', 'MU', 'ANET', 'AAPL', 'MMM', 'CAT']  # 처리할 목록
result_df = run_batch_valuation(
    tickers=tickers,
    api_key='hT0gAk87j9xZx4PlBApvBqfVL5IahvgV',
    db_info=db_info,
    start_date_month='2011-03-01',
    end_date_month=(pd.Timestamp.today().normalize() - MonthEnd(1)).strftime('%Y-%m-%d'),
    min_rows_required=44
)
print(result_df.tail())
# 이제 result_df를 DB 저장 등에 활용하시면 됩니다.

Using project path: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy


ImportError: cannot import name 'fetch_revenue_data' from 'DATA.stock_invest_function' (C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA\stock_invest_function.py)

In [15]:
import sys, os
from pathlib import Path
import pandas as pd
import numpy as np
from typing import Optional
# from stock_forecast.Korea_Market.valuation.kse_valuation_machine_v1 import psr_forecast_df


def add_repo_path():
    here = Path.cwd()
    # 현재 위치부터 상위 폴더를 훑으며 DATA 폴더가 보이는 지점 찾기
    for p in [here, *here.parents]:
        if (p / "DATA").exists():
            if str(p) not in sys.path:
                sys.path.insert(0, str(p))
            return str(p)
    # 못 찾으면 로컬 고정 경로(본인 PC 경로로) 마지막 보루로 추가
    fallback = r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast"
    if os.path.isdir(fallback) and fallback not in sys.path:
        sys.path.insert(0, fallback)
    return fallback

project_path = add_repo_path()
print("Using project path:", project_path)

import calendar
import time
# from DATA.stock_invest_function import *
from DATA.stock_invest_function import *
from datetime import datetime
from dateutil.relativedelta import relativedelta
warnings.filterwarnings('ignore')

import importlib
import DATA.us_sarima_forecast as sarima
importlib.reload(sarima)
import DATA.us_lstm_forecast_v2 as lstm_v2
importlib.reload(lstm_v2)
import DATA.us_prophet_forecast_v3 as prophet_v3
importlib.reload(prophet_v3)
import DATA.us_est_forecast_v2 as esmod
importlib.reload(esmod)

Using project path: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy


In [16]:
# 유틸리티 함수들
def convert_to_month_end(date_str):
    try:
        # 문자열/타입 혼용 안전 변환
        date_obj = pd.to_datetime(date_str)
        if pd.isna(date_obj):
            return None

        y, m, d = date_obj.year, date_obj.month, date_obj.day

        # 1~5일 → 전달 말일
        if 1 <= d <= 5:
            if m == 1:
                prev_y, prev_m = y - 1, 12
            else:
                prev_y, prev_m = y, m - 1
            last_day_prev = calendar.monthrange(prev_y, prev_m)[1]
            return datetime(prev_y, prev_m, last_day_prev)

        # 그 외 → 해당월 말일
        last_day_cur = calendar.monthrange(y, m)[1]
        return datetime(y, m, last_day_cur)

    except Exception:
        return None

def process_daily_to_monthly_market_data(daily_data, ticker):
    if not daily_data:
        return pd.DataFrame()
    df = pd.DataFrame(daily_data)
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date')
    df['year_month'] = df['date'].dt.to_period('M')
    monthly_data = []
    for year_month in df['year_month'].unique():
        month_data = df[df['year_month'] == year_month]
        last_day_data = month_data.loc[month_data['date'].idxmax()]
        monthly_data.append({
            'ticker': ticker,
            'date': last_day_data['date'],
            'market_cap': last_day_data['marketCap'],
            'market_cap_billions': round(last_day_data['marketCap'] / 1_000_000_000, 2),
        })
    return pd.DataFrame(monthly_data)


def fetch_revenue_data(ticker, api_key):
    url = f"https://financialmodelingprep.com/api/v3/income-statement/{ticker}"
    params = {'limit': 200, 'apikey': api_key, 'period': 'quarter'}
    try:
        response = requests.get(url, params=params, timeout=30)
        if response.status_code != 200:
            return None, f"HTTP {response.status_code}"
        data = response.json()
        if isinstance(data, dict) and 'Error Message' in data:
            return None, f"API 오류: {data['Error Message']}"
        if not data:
            return None, "데이터 없음"
        return data, None
    except Exception as e:
        return None, f"오류: {str(e)}"

def fetch_market_data_yearly(ticker, api_key, start_year=2010):
    all_data = []
    current_year = datetime.now().year
    for year in range(start_year, current_year + 1):
        start_date_str = f"{year}-01-01"
        end_date_str = f"{year}-12-31"
        url = f"https://financialmodelingprep.com/api/v3/historical-market-capitalization/{ticker}"
        params = {'from': start_date_str, 'to': end_date_str, 'apikey': api_key}
        try:
            response = requests.get(url, params=params, timeout=30)
            if response.status_code == 200:
                data = response.json()
                if data and isinstance(data, list):
                    all_data.extend(data)
            time.sleep(0.3)
        except Exception as e:
            continue
    return all_data if all_data else None, None

def fetch_db_revenue_data(ticker, db_info, end_date='2025-08-31'):
    try:
        engine = create_engine(
            f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
            f"{db_info['host']}:{db_info['port']}/{db_info['database']}"
        )
        query = f"""
        SELECT date, ticker, saleq
        FROM US_fundq
        WHERE ticker = '{ticker}'
        AND saleq IS NOT NULL
        AND date <= '{end_date}'
        ORDER BY date ASC
        """
        df = pd.read_sql(query, con=engine)
        engine.dispose()
        if not df.empty:
            df['date'] = pd.to_datetime(df['date'])
            df['revenue_billions'] = df['saleq'] / 1000
            df['date_month_end'] = df['date'].apply(convert_to_month_end)
        return df[['ticker', 'date', 'date_month_end', 'revenue_billions']]
    except Exception as e:
        return pd.DataFrame()

def fetch_db_market_data(ticker, db_info, end_date='2024-12-31'):
    try:
        engine = create_engine(
            f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
            f"{db_info['host']}:{db_info['port']}/{db_info['database']}"
        )
        query = f"""
        SELECT date, ticker, me
        FROM US_fundm
        WHERE ticker = '{ticker}'
        AND me IS NOT NULL
        AND date <= '{end_date}'
        ORDER BY date ASC
        """
        df = pd.read_sql(query, con=engine)
        engine.dispose()
        if not df.empty:
            df['date'] = pd.to_datetime(df['date'])
            df['market_cap_billions'] = df['me'] / 1000
            df['date_month_end'] = df['date'].apply(convert_to_month_end)
        return df[['ticker', 'date', 'date_month_end', 'market_cap_billions']]
    except Exception as e:
        return pd.DataFrame()

def calculate_enhanced_ttm_and_psr(merged_data):
    """Calculate enhanced TTM and PSR"""
    df = merged_data.copy()

    # ✅ 날짜형으로 변환 (핵심 수정)
    df['date_month_end'] = pd.to_datetime(df['date_month_end'], errors='coerce')
    df = df.sort_values(['date_month_end']).reset_index(drop=True)
    df = df.sort_values(['ticker', 'date_month_end']).reset_index(drop=True)

    # Calculate TTM from quarterly revenue
    df['revenue_ttm'] = df.groupby('ticker')['revenue_billions'].rolling(window=4, min_periods=1).sum().reset_index(0,
                                                                                                                    drop=True)
    df['revenue_ttm_billions'] = df['revenue_ttm']

    # Apply 2-month shift
    df['revenue_ttm_shift'] = df.groupby('ticker')['revenue_ttm_billions'].shift(2)

    # Calculate PSR
    df['PSR_ttm'] = df['market_cap_billions'] / df['revenue_ttm_shift']

    # Handle infinite values
    df['PSR_ttm'] = df['PSR_ttm'].replace([np.inf, -np.inf], np.nan)

    return df

def prepare_revenue_ttm(
    df: pd.DataFrame,
    revenue_key: str = "revenue_billions",
    min_periods: int = 1,   # 완전한 TTM만 원하면 4로 바꾸세요
) -> pd.DataFrame:
    """
    1) revenue 칼럼들의 NaN을 '해당 행의 revenue 평균'으로 채움
    2) 각 revenue 칼럼의 4분기 합(TTM)을 *_ttm 칼럼으로 생성 (시차 없음)
    - 그룹 기준: ticker
    - 정렬 기준: date_month_end (월말 날짜)
    """
    d = df.copy()

    # --- 키 정리 ---
    # date_month_end: index에 있으면 칼럼으로 복구
    if 'date_month_end' not in d.columns:
        d = d.reset_index().rename(columns={'index': 'date_month_end'})
    d['date_month_end'] = pd.to_datetime(d['date_month_end'])

    if 'ticker' not in d.columns:
        raise ValueError("ticker 칼럼이 필요합니다.")

    # --- revenue 칼럼 자동 탐지 ---
    rev_cols = [c for c in d.columns if revenue_key in c]
    if not rev_cols:
        raise ValueError(f"'{revenue_key}' 가 포함된 칼럼을 찾지 못했습니다.")

    # --- ticker NaN 보정 ---
    # 단일 티커면 ffill/bfill로 채움, 복수 티커면 NaN 행 제거(필요 시 정책 조정)
    uniq_tickers = d['ticker'].dropna().unique()
    if len(uniq_tickers) == 1:
        d['ticker'] = d['ticker'].ffill().bfill()
    else:
        d = d[~d['ticker'].isna()].copy()

    # --- 정렬 ---
    d = d.sort_values(['ticker', 'date_month_end']).reset_index(drop=True)

    # --- NaN 보간: 행 단위 평균으로 revenue 결측치 채우기 ---
    row_mean = d[rev_cols].mean(axis=1, skipna=True)
    for c in rev_cols:
        d[c] = d[c].fillna(row_mean)

    # --- TTM 계산 (최근 4분기 합, 시차 없음) ---
    for c in rev_cols:
        ttm_col = f"{c}_ttm"
        d[ttm_col] = (
            d.groupby('ticker', group_keys=False)[c]
             .rolling(window=4, min_periods=min_periods)
             .sum()
             .reset_index(level=0, drop=True)
        )

    d = d.set_index('date_month_end')

    return d

def clean_rev_data(rev_data: pd.DataFrame) -> pd.DataFrame:
    """
    1) 'revenue' 컬럼 값이 NaN인 행 제거
    2) (calendar_year, period) 중복 행 제거 (첫 번째 행만 유지)
       - 입력 순서를 그대로 기준으로 '첫째 데이터'를 보존
    """
    required = ['revenue', 'calendar_year', 'period']
    missing = [c for c in required if c not in rev_data.columns]
    if missing:
        raise ValueError(f"필수 컬럼이 없습니다: {missing}")

    d = rev_data.copy()

    # 1) revenue NaN인 행 제거
    before = len(d)
    d = d[~d['revenue'].isna()].copy()
    removed_nan = before - len(d)

    # 2) (calendar_year, period) 중복 제거 — 첫 행 유지(현재 순서 기준)
    before2 = len(d)
    d = d.drop_duplicates(subset=['calendar_year', 'period'], keep='first').reset_index(drop=True)
    removed_dup = before2 - len(d)

    print(f"[clean_rev_data_minimal] removed rows → revenue NaN: {removed_nan}, duplicates: {removed_dup}")
    return d

# ==========================
# 사용 예시
# ==========================
# cleaned = clean_rev_data(rev_data)
# cleaned.head()


In [180]:
# 설정값들
ticker = 'VVV'

hs_code = '841191'

api_key = 'hT0gAk87j9xZx4PlBApvBqfVL5IahvgV'

db_info = {
    'host': get_db_host(),
    # 'host': '192.168.0.230',
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

start_date_month = '2011-03-01'
end_date_month = (pd.Timestamp.today().normalize() - pd.offsets.MonthEnd(1)).strftime('%Y-%m-%d')

In [181]:
print("=" * 80)
print("전처리 과정 테스트 시작")
print(f"대상 종목: {ticker}")
print("=" * 80)

# 1. FMP 매출 데이터 수집
print("\n1. FMP 매출 데이터 수집 중...")
revenue_data, error = fetch_revenue_data(ticker, api_key)

if revenue_data is None:
    print(f"ERROR: FMP 매출 데이터 수집 실패 - {error}")
    exit()

all_revenue_data = []
for item in revenue_data:
    all_revenue_data.append({
        'ticker': ticker,
        'date': item.get('date', ''),
        'calendar_year': item.get('calendarYear', ''),
        'period': item.get('period', ''),
        'revenue': item.get('revenue', 0) if item.get('revenue') is not None else 0,
        'revenue_billions': round((item.get('revenue', 0) or 0) / 1_000_000_000, 2),
    })

fmp_revenue_df = pd.DataFrame(all_revenue_data)
fmp_revenue_df['date'] = pd.to_datetime(fmp_revenue_df['date'])
fmp_revenue_df = fmp_revenue_df.sort_values(['ticker', 'date'])
fmp_revenue_df['date_month_end'] = fmp_revenue_df['date'].apply(convert_to_month_end)
fmp_revenue_df = fmp_revenue_df.drop_duplicates(subset=['date_month_end'], keep='first').reset_index(drop=True)
print(f"FMP 매출 데이터: {len(fmp_revenue_df)}건")


# 2. DB 매출 데이터 가져오기
db_revenue_raw = fetch_db_revenue_data(ticker, db_info)
db_revenue_df = db_revenue_raw.loc[db_revenue_raw['revenue_billions'] != db_revenue_raw['revenue_billions'].shift()]

# db_revenue_df = db_revenue_raw.drop_duplicates(subset=['date_month_end', 'revenue_billions'], keep='first')
mereged_rev_data = pd.merge(fmp_revenue_df, db_revenue_df, on = ['ticker', 'date_month_end'], how='outer')

rev_data = mereged_rev_data[mereged_rev_data['date_month_end'] >= start_date_month ]
rev_data['revenue_billions_x'] = rev_data['revenue_billions_x'].fillna(rev_data['revenue_billions_y'])
# 컬럼 이름 변경
rev_data.rename(columns={'revenue_billions_x': 'revenue_billions'}, inplace=True)

rev_data = clean_rev_data(rev_data)

전처리 과정 테스트 시작
대상 종목: VVV

1. FMP 매출 데이터 수집 중...
FMP 매출 데이터: 43건
[clean_rev_data_minimal] removed rows → revenue NaN: 0, duplicates: 0


In [183]:
# periods=4 또는 8 등 원하는 분기 수
periods = 4

# 모듈 함수 호출 (내부에서 월말 정렬/중복제거 처리)
sarima_df, results = sarima.run_sarima_prediction(
    rev_data,
    forecast_quarters=periods,   # ← 예측 분기 수
    exog_col=None                # 외생변수 없으면 None
)

# 인덱스를 date_month_end로 설정
sarima_df = sarima_df.sort_values("date_month_end").set_index("date_month_end")

# 1) 4분기 예측
# 4분기 예측
lstm_raw_df, lstm_results_4q = lstm_v2.run_lstm_revenue_prediction(rev_data, ticker=ticker, prediction_quarters=4)
lstm_df = lstm_raw_df.drop_duplicates(subset=['revenue_billions_lstm_forecast'], keep='last')

# 4분기 예측
prophet_raw_df, res_4q = prophet_v3.run_prophet_revenue_only(rev_data, ticker=ticker, prediction_quarters=4)

# 4분기 예측
es_raw_df, res_q4 = esmod.run_es_revenue_quarterly(rev_data, ticker=ticker, prediction_quarters=4)
# es_raw_df.tail(24)

In [188]:
# 3. FMP 시가총액 데이터 수집
print("2. FMP 시가총액 데이터 수집 중...")
market_data, error = fetch_market_data_yearly(ticker, api_key, start_year=2010)

if not market_data:
    print("ERROR: FMP 시가총액 데이터 수집 실패")
    raise SystemExit(1)

fmp_market_df = process_daily_to_monthly_market_data(market_data, ticker).copy()
fmp_market_df['date_month_end'] = fmp_market_df['date'].apply(convert_to_month_end)
# 혹시 중복/정렬 문제 예방
fmp_market_df = (fmp_market_df
                 .drop_duplicates(subset=['date_month_end'])
                 .sort_values('date_month_end')
                 .reset_index(drop=True))

print(f"FMP 시가총액 데이터: {len(fmp_market_df)}건")

# -----------------------------
# 안전 병합: DB가 없으면 FMP만 사용
# -----------------------------
def _safe_get_db_market_df():
    try:
        df = fetch_db_market_data(ticker, db_info)
        # None 이거나 길이 0이면 빈 DF 반환
        if df is None or len(df) == 0:
            return pd.DataFrame()
        return df.copy()
    except Exception as e:
        print(f"[WARN] DB 조회 중 예외 발생: {e}")
        return pd.DataFrame()

db_market_df = _safe_get_db_market_df()

# DB가 있으면 date_month_end 정규화 + 컬럼 정리
if not db_market_df.empty:
    # 날짜 컬럼 유도: date_month_end가 없고 date가 있으면 생성
    if 'date_month_end' not in db_market_df.columns:
        if 'date' in db_market_df.columns:
            db_market_df['date_month_end'] = db_market_df['date'].apply(convert_to_month_end)
        else:
            # 날짜 정보가 없으면 병합 불가 → 빈 DF 취급
            print("[WARN] DB 데이터에 날짜 컬럼이 없어 병합을 건너뜁니다.")
            db_market_df = pd.DataFrame()

if db_market_df.empty:
    # DB가 비어 있으면 FMP만 사용
    print("[INFO] DB 시가총액 데이터 없음 → FMP 데이터만 사용합니다.")
    merged_market_df = fmp_market_df.copy()
    # from_db 컬럼은 NaN으로 생성(분석 시 출처 구분 유용)
    merged_market_df['market_cap_billions_from_db'] = np.nan

else:
    # 필요한 컬럼명 정리
    # DB에 market_cap_billions가 있으면 rename, 없으면 NaN으로 준비
    if 'market_cap_billions' in db_market_df.columns:
        db_market_df_renamed = db_market_df.rename(
            columns={'market_cap_billions': 'market_cap_billions_from_db'}
        )
    else:
        # 필요한 최소 컬럼만 추려서 NaN 채우기
        db_market_df_renamed = db_market_df[['date_month_end']].copy()
        db_market_df_renamed['market_cap_billions_from_db'] = np.nan
        print("[WARN] DB에 'market_cap_billions' 컬럼이 없어 NaN으로 채웁니다.")

    # 병합 (분기/월말 정렬 맞춤)
    merged_market_df = fmp_market_df.merge(
        db_market_df_renamed[['date_month_end', 'market_cap_billions_from_db']],
        on='date_month_end',
        how='left'   # FMP 기준으로 맞추고 DB 값 있으면 붙임
    )

# 최종 결측 보충: FMP 값이 NaN이면 DB 값으로 대체
if 'market_cap_billions' not in merged_market_df.columns:
    # 혹시 FMP 가 다른 이름을 썼다면 여기서 보정하세요.
    # 일단 없으면 새로 만들고 DB로 채움
    merged_market_df['market_cap_billions'] = np.nan

if 'market_cap_billions_from_db' not in merged_market_df.columns:
    merged_market_df['market_cap_billions_from_db'] = np.nan

merged_market_df['market_cap_billions'] = merged_market_df['market_cap_billions'].fillna(
    merged_market_df['market_cap_billions_from_db']
)

# 정리
merged_market_df = (merged_market_df
                    .drop_duplicates(subset=['date_month_end'])
                    .sort_values('date_month_end')
                    .reset_index(drop=True))

print(f"병합 완료: {len(merged_market_df)}건 (FMP+DB)")


2. FMP 시가총액 데이터 수집 중...
FMP 시가총액 데이터: 109건
[INFO] DB 시가총액 데이터 없음 → FMP 데이터만 사용합니다.
병합 완료: 109건 (FMP+DB)


In [189]:
# merged_market_df

enhanced_merged_df = pd.merge(merged_market_df[['date_month_end', 'market_cap_billions']], rev_data, on='date_month_end', how='outer')
market_cap_resize = enhanced_merged_df[['date_month_end', 'market_cap_billions', 'ticker', 'revenue_billions']].copy()
market_cap_resize.dropna(subset =['market_cap_billions'], inplace=True)
market_cap_resize.ffill(limit=2, inplace=True)
market_cap_resize = market_cap_resize[(market_cap_resize['date_month_end'] >= start_date_month ) & (market_cap_resize['date_month_end'] <= end_date_month)]

market_cap_resize = market_cap_resize.dropna(axis=0)
# market_cap_resize
enhanced_merged_df_with_ttm = calculate_enhanced_ttm_and_psr(market_cap_resize)

from DATA.us_sarima_forecast import run_sarima_psr_only

# 12개월 예측
psr_sarima_df, psr_12_res = run_sarima_psr_only(
    df=enhanced_merged_df_with_ttm,                 # date_month_end, PSR_ttm 포함
    periods=12,                  # 12개월
    target_col="PSR_ttm",        # 다른 월간 변수로 교체 가능
    analysis_start="2012-06-01", # 2012년 이후만 분석
    warmup_months=6,             # 최초 유효값 + 6개월부터 학습
    fill_method="interpolate",   # 보간 후 ffill/bfill
    ic="aic"
)

# df: 최소 ['date_month_end','PSR_ttm'] 포함, 가능하면 보조피처도 포함
psr_lstm_df, psr_results = lstm_v2.run_lstm_psr_prediction(enhanced_merged_df_with_ttm, ticker=ticker, prediction_months=12)
# psr_df에는 'PSR_ttm_lstm_forecast' 컬럼이 추가됩니다.

# 1) 자동 start_date (데이터 마지막 월 다음 달부터)
psr_prophet_df, psr_res = prophet_v3.run_prophet_psr_only(enhanced_merged_df_with_ttm, ticker=ticker, prediction_months=12)

psr_es_df, psr_res_es = esmod.run_es_psr_only(
    df=enhanced_merged_df_with_ttm,  # 반드시 date_month_end / PSR_ttm 포함
    ticker= ticker,
    prediction_months=12,
    start_date=None  # None이면 자동: (데이터 max) + 1개월 말일부터
)

In [195]:
#### 4. Valuation 종합
sarima_resize_df = sarima_df[['ticker', 'revenue_billions_sarima_noexog']].copy()
lstm_resize_df = lstm_df[[ 'revenue_billions_lstm_forecast']].copy()
prophet_resize_df = prophet_raw_df[['revenue_billions_prophet_forecast']].copy()
es_resize_df = es_raw_df[['revenue_billions_esq_forecast']].copy()

revenue_forecast_df = pd.concat([sarima_resize_df, lstm_resize_df, prophet_resize_df, es_resize_df], axis=1)

psr_sarima_resiae = psr_sarima_df[['PSR_ttm_sarima_forecast']]
psr_lstm_resiae = psr_lstm_df[['PSR_ttm_lstm_forecast']]
psr_prophet_resiae = psr_prophet_df[['PSR_prophet_forecast_noexog']]
psr_es_resiae = psr_es_df[['PSR_es_forecast']]

psr_forecast_df = pd.concat([psr_sarima_resiae, psr_lstm_resiae, psr_prophet_resiae,  psr_es_resiae], axis =1)

revenue_forecast_ = prepare_revenue_ttm(revenue_forecast_df)
revenue_forecast_ttm = revenue_forecast_.filter(like = '_ttm')
revenue_forecast_ttm['ticker'] = ticker

valuation_df = pd.concat([revenue_forecast_ttm, psr_forecast_df], axis=1)

# 1) 복사본 생성 (원본 보호)
valuation_filled = valuation_df.copy()

# 2) ffill 대상 칼럼 목록 생성
cols_to_fill = ['ticker'] + [c for c in valuation_filled.columns if 'revenue' in c]

# 3) 선택된 칼럼만 ffill(limit=2)
valuation_filled[cols_to_fill] = valuation_filled[cols_to_fill].ffill(limit=2)

required_cols = valuation_filled.columns.tolist()

missing = [c for c in required_cols if c not in valuation_filled.columns]
if missing:
    raise ValueError(f"다음 칼럼이 없습니다: {missing}")

# 2. Valuation 계산 (revenue × PSR)
valuation_filled['sarima_valuation'] = (
    valuation_filled['revenue_billions_sarima_noexog_ttm'] *
    valuation_filled['PSR_ttm_sarima_forecast']
)

valuation_filled['lstm_valuation'] = (
    valuation_filled['revenue_billions_lstm_forecast_ttm'] *
    valuation_filled['PSR_ttm_lstm_forecast']
)

valuation_filled['prophet_valuation'] = (
    valuation_filled['revenue_billions_prophet_forecast_ttm'] *
    valuation_filled['PSR_prophet_forecast_noexog']
)

valuation_filled['es_valuation'] = (
    valuation_filled['revenue_billions_esq_forecast_ttm'] *
    valuation_filled['PSR_es_forecast']
)

# 3. 마지막 15개월 추출
# (date 칼럼이 없다면, 대신 index가 날짜인 경우로 가정)
if 'date_month_end' in valuation_filled.columns:
    valuation_filled = valuation_filled.sort_values('date_month_end')
    valuation_result = valuation_filled.groupby('ticker').tail(15).reset_index(drop=True)
else:
    # index가 날짜라고 가정
    valuation_filled = valuation_filled.sort_index()
    valuation_result = valuation_filled.groupby('ticker').tail(15).reset_index()

In [199]:
valuation_result

,index,revenue_billions_sarima_noexog_ttm,revenue_billions_lstm_forecast_ttm,revenue_billions_prophet_forecast_ttm,revenue_billions_esq_forecast_ttm,ticker,PSR_ttm_sarima_forecast,PSR_ttm_lstm_forecast,PSR_prophet_forecast_noexog,PSR_es_forecast,sarima_valuation,lstm_valuation,prophet_valuation,es_valuation
0,2025-06-30,1.690000,1.690000,1.690000,1.690000,VVV,2.981481,2.981481,2.981481,2.981481,5.038704,5.038704,5.038704,5.038704
1,2025-07-31,1.690000,1.690000,1.690000,1.690000,VVV,2.795031,2.795031,2.795031,2.795031,4.723602,4.723602,4.723602,4.723602
2,2025-08-31,1.690000,1.690000,1.690000,1.690000,VVV,3.018293,3.018293,3.018293,3.018293,5.100915,5.100915,5.100915,5.100915
3,2025-09-30,1.682054,1.614537,1.773354,1.685661,VVV,2.999080,2.976185,3.193123,3.115601,5.044614,4.805160,5.662537,5.251847
4,2025-10-31,1.682054,1.614537,1.773354,1.685661,VVV,2.980101,2.924288,3.095619,3.241213,5.012689,4.721370,5.489628,5.463586
5,2025-11-30,1.682054,1.614537,1.773354,1.685661,VVV,2.961349,2.880115,4.511686,3.366825,4.981148,4.650052,8.000816,5.675326
6,2025-12-31,1.696304,1.568016,1.876565,1.710051,VVV,2.942824,2.847328,3.714398,3.492437,4.991924,4.464657,6.970309,5.972247
7,2026-01-31,1.696304,1.568016,1.876565,1.710051,VVV,2.924520,2.811338,3.806350,3.618049,4.960875,4.408224,7.142863,6.187051
8,2026-02-28,1.696304,1.568016,1.876565,1.710051,VVV,2.906435,2.793387,3.911881,3.743662,4.930197,4.380077,7.340898,6.401854
9,2026-03-31,1.712893,1.530433,1.991077,1.743171,VVV,2.888565,2.773056,3.899757,3.869274,4.947802,4.243976,7.764715,6.744806


In [201]:
last_mkt_cap = fmp_market_df.tail(10)

,ticker,date,market_cap,market_cap_billions,date_month_end
99,VVV,2024-12-31,5000076000,5.00,2024-12-31
100,VVV,2025-01-31,5128602000,5.13,2025-01-31
101,VVV,2025-02-28,4746455963,4.75,2025-02-28
102,VVV,2025-03-31,4480046965,4.48,2025-03-31
103,VVV,2025-04-30,4409262000,4.41,2025-04-30
104,VVV,2025-05-30,4413684000,4.41,2025-05-31
105,VVV,2025-06-30,4832212000,4.83,2025-06-30
106,VVV,2025-07-31,4497900000,4.50,2025-07-31
107,VVV,2025-08-29,4948328000,4.95,2025-08-31
108,VVV,2025-09-30,4582116000,4.58,2025-09-30
